In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
data = pd.read_csv('../data/housing.csv')

In [3]:
data.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='object')

In [8]:
data['median_income'].describe()

count    20640.000000
mean         3.870671
std          1.899822
min          0.499900
25%          2.563400
50%          3.534800
75%          4.743250
max         15.000100
Name: median_income, dtype: float64

In [12]:
data['income_category'] = pd.cut(data['median_income'], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5])

In [13]:
data.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity', 'income_category'],
      dtype='object')

In [14]:
data.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income_category
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,5
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,5
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,5
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,4
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,3


In [29]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(data, test_size=0.2, stratify=data['income_category'], random_state=120)

In [33]:
print(len(data), len(train_set), len(test_set))

20640 16512 4128


# EDA

In [36]:
from ydata_profiling import ProfileReport

profile = ProfileReport(train_set, title="Housing Data Profiling Report")
profile.to_file("housing_data_profile.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 11/11 [00:00<00:00, 304.30it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import dtale
d = dtale.show(train_set)
d.open_browser()

/home/meftaul/miniconda3/envs/e2e-ml/lib/python3.10/site-packages/dtale/column_analysis.py:402: FutureWarning:

pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.

/home/meftaul/miniconda3/envs/e2e-ml/lib/python3.10/site-packages/dtale/column_analysis.py:402: FutureWarning:

pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.



In [38]:
d.kill()

ERROR	Thread(Thread-143 (process_request_thread)) dtale.app:app.py:shutdown_server()- weakly-referenced object no longer exists
ERROR	Thread(Thread-143 (process_request_thread)) dtale.app:app.py:shutdown_server()- weakly-referenced object no longer exists


# Building the ML Model

In [51]:
X = train_set.drop(['income_category', 'median_house_value'], axis=1)
y = train_set[['median_house_value']]

In [57]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

numeric_features = X[numeric_cols]
categorical_features = X[categorical_cols]

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Numeric features shape:", numeric_features.shape)
print("Categorical features shape:", categorical_features.shape)

Numeric columns: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
Categorical columns: ['ocean_proximity']
Numeric features shape: (16512, 8)
Categorical features shape: (16512, 1)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn import set_config
set_config(display='diagram')

In [79]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])


preprocessing_pipeline = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])


lin_reg_model = Pipeline([
    ('preprocessor', preprocessing_pipeline),
    ('model', LinearRegression())
])

random_forest_model = Pipeline([
    ('preprocessor', preprocessing_pipeline),
    ('model', RandomForestRegressor(random_state=120))
])

# random_forest_model

# lin_reg_model

In [80]:
# create train and validation set from X and y
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=120)

lin_model = lin_reg_model.fit(X_train, y_train)
rf_model = random_forest_model.fit(X_train, y_train)

/home/meftaul/miniconda3/envs/e2e-ml/lib/python3.10/site-packages/sklearn/base.py:1365: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().



In [81]:
from sklearn.metrics import r2_score

# a method that takes a model and a dataset and returns the r2 score and rmse
def evaluate_model(model, data, labels):
    predictions = model.predict(data)
    r2 = r2_score(labels, predictions)
    return {'r2': r2}

In [82]:
print("Linear Regression Evaluation:", evaluate_model(lin_model, X_val, y_val))

print("Random Forest Evaluation:", evaluate_model(rf_model, X_val, y_val))

Linear Regression Evaluation: {'r2': 0.6343917097007563}
Random Forest Evaluation: {'r2': 0.8157834738819572}


In [84]:
import joblib

joblib.dump(rf_model, '../model/forest_model.pkl')

['../model/forest_model.pkl']

In [85]:
test_set.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income_category
20471,-118.77,34.26,26.0,3038.0,468.0,1825.0,468.0,5.6385,196900.0,<1H OCEAN,4
15638,-122.41,37.79,52.0,3598.0,1011.0,2062.0,966.0,2.9871,380000.0,NEAR BAY,2
1022,-119.78,38.69,17.0,1364.0,282.0,338.0,152.0,2.4500,117600.0,INLAND,2
12596,-121.48,38.53,43.0,1378.0,280.0,708.0,280.0,2.3542,103900.0,INLAND,2
14793,-117.12,32.57,35.0,1450.0,256.0,930.0,286.0,2.6715,133300.0,NEAR OCEAN,2


In [86]:
X_test = test_set.drop(['income_category', 'median_house_value'], axis=1)
y_test = test_set[['median_house_value']]

In [87]:
rf_model_loaded = joblib.load('../model/forest_model.pkl')

print("Random Forest Evaluation on Test Set:", evaluate_model(rf_model_loaded, X_test, y_test))

Random Forest Evaluation on Test Set: {'r2': 0.8158767758066197}
